# DA6401 — System 3: Segmentation (full fine-tune)

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Quick Save** to run in background (~5 hrs)

**What this notebook does:**
- Quick classifier training (80 epochs — just to get pretrained encoder weights)
- §2.3 Transfer learning: full fine-tune (all encoder layers trainable)
- §2.6 Segmentation masks (auto-logged every 10 epochs)

**W&B tag:** `kaggle-sys3`

**Output files:**
- `checkpoints/classifier.pth` — encoder-only classifier (80 ep)
- `checkpoints/unet.pth` — full fine-tune segmentation model
- `checkpoints/unet_full.pth` — backup copy

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -2
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print(f"GPU: {gpu}")
assert torch.cuda.is_available(), "NO GPU — go to Settings and enable P100!"

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

## 1. Quick Classifier Training (encoder pretraining)

We need a trained encoder for the segmentation task. 80 epochs is enough to get good encoder features — this is NOT one of the main dropout experiments (those run in Sys1 and Sys2).

In [ ]:
# ── Quick encoder pretraining (80 epochs, just for encoder weights) ──
!python train.py --task classify --experiment kaggle-sys3-pretrain --dropout 0.5 \
    --epochs 80 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

print("\n--- Encoder ready → classifier.pth will be used for segmentation below ---")

## 2. Segmentation — Full Fine-tune (§2.3)

All encoder layers are trainable (full fine-tuning). This completes the 3-way transfer learning comparison:
- Frozen → Sys2
- Partial → Sys2
- **Full → this notebook**

In [ ]:
# ── §2.3  Segmentation: FULL fine-tune (all layers trainable) ──
!python train.py --task segment --experiment kaggle-sys3 --freeze-mode full \
    --epochs 120 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

# Backup
!cp {CKPT}/unet.pth {CKPT}/unet_full.pth
print("\n--- Backed up → unet_full.pth ---")

## 3. Summary

In [ ]:
import os

CKPT = "/kaggle/working/checkpoints"

print("="*60)
print("  SYSTEM 3 — FINAL SUMMARY")
print("="*60)
print("\n  Files in /kaggle/working/checkpoints/:")
for f in sorted(os.listdir(CKPT)):
    size_mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"    {f:40s} {size_mb:6.1f} MB")

print("\n  W&B runs logged under tag: kaggle-sys3")
print("  Sections covered: §2.3 (Transfer: full fine-tune), §2.6 (Seg masks)")
print("\n  Download checkpoints from the Output tab after session ends!")
print("="*60)